# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
Croissant schema URL:
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load the dataset Croissant metadata and explore core attributes.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant JSON-LD URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"Description: {getattr(metadata, 'description', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")

## 2. Data Overview
Examine available RecordSets, Fields, and related IDs (`@id`).

`mlcroissant` allows us to inspect the structure using record sets and fields. Each entity has an `@id` that is used for unambiguous referencing.

In [ ]:
# List all record sets in the dataset (referenced by @id)
print("Record Sets found in the dataset:")
record_sets = list(dataset.record_sets)
for record_set in record_sets:
    print(f"- {record_set['@id']}: {record_set.get('name', '')}")

# For each record set, show fields (with their @id)
for record_set in record_sets:
    print(f"\nRecordSet @id: {record_set['@id']} | Name: {record_set.get('name', '')}")
    fields = record_set.get('fields', [])
    if not fields:
        print('  (No fields found)')
    else:
        for field in fields:
            print(f"  - Field @id: {field['@id']:<36} | Name: {field.get('name', '')}")

## 3. Data Extraction
We'll load data from a chosen record set using its `@id`.

All fields and record sets are referenced by their `@id` for reproducibility.

In [ ]:
# Choose the main protein data RecordSet by its @id.
# In this dataset, the table of experimental results is typically labeled with 'dv:ExperimentDataset' or similar.
# Let's list candidate record sets again to ensure correctness.
record_set_ids = [r["@id"] for r in record_sets]
print("Candidate record set @id's:", record_set_ids)

# We'll use the first RecordSet if available, or specify explicitly.
main_recordset_id = record_set_ids[0] if record_set_ids else None  # e.g., 'dv:ExperimentDataset'

# Inspect first few records for that record set
records = list(dataset.records(record_set=main_recordset_id))
df = pd.DataFrame(records)

print(f"Loaded {len(df)} records. Columns found:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Let's perform basic filtering, normalization, and grouping operations based on the dataset. All field references are always by `@id`.

In [ ]:
# Preview columns to select meaningful numeric and grouping fields
print(f"Columns: {df.columns.tolist()}")
# Example IDs (update if necessary for this schema)
# Use field names such as 'cr:abundance', 'cr:MW', etc. if these @ids exist as column headers

# Try to choose a numeric field present in the dataset:
candidate_numeric_fields = [col for col in df.columns if df[col].dtype in ['int64', 'float64'] or pd.api.types.is_numeric_dtype(df[col])]
if candidate_numeric_fields:
    numeric_field = candidate_numeric_fields[0]  # Use the first found
    print(f"Choosing numeric field for analysis: {numeric_field}")
else:
    # Fall back to a string known commonly in such datasets (to update as needed for custom schemas)
    numeric_field = None

# Set a threshold (e.g., greater than the mean)
if numeric_field:
    threshold = df[numeric_field].mean()
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records where {numeric_field} > {threshold:.2f} ({len(filtered_df)} records)")
    # Normalize
    normalized_col = f"{numeric_field}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(filtered_df[[numeric_field, normalized_col]].head())
    # Try grouping on a categorical field (e.g. 'cr:experiment_group' if present)
    candidate_group_fields = [col for col in df.columns if (df[col].dtype == object or pd.api.types.is_categorical_dtype(df[col])) and col != numeric_field]
    group_field = candidate_group_fields[0] if candidate_group_fields else None
    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"Grouped mean of {numeric_field} by {group_field}:")
        print(grouped_df.head())
else:
    print("No numeric field found to perform EDA. Please check field names and types.")

## 5. Visualization
We'll plot the distribution of the chosen numeric field, and if there's a grouping field, visualize by group as well.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and not df[numeric_field].isnull().all():
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field], kde=True, bins=30, color='steelblue')
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()
    
    if group_field is not None:
        plt.figure(figsize=(10, 4))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} grouped by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field with data found for visualization.")

## 6. Conclusion
- We demonstrated how to load a Croissant-formatted FAIR dataset with `mlcroissant` using only `@id` references for all entities.
- The Mass Spectrometry dataset provides quantitative measurements about proteins from stimulated human mast cells.
- Exploratory data analysis and visualizations revealed basic trends and how to group/filter/norm values for downstream bioinformatics and ML studies.

_**For formal analysis, consult the dataset schema for detailed @id mappings and the Sen.Science FAIR portal documentation.**_